# CyberSOC GRPO Training

Trains a SOC-analyst LLM using **Group Relative Policy Optimization** directly against
`CyberSOCEnvironment` (no HTTP server required — the env runs in-process).

| GPU | VRAM | Auto-selected model |
|-----|------|--------------------|
| T4  | 15 GB | Qwen2.5-3B-Instruct-bnb-4bit |
| A10G | 24 GB | Qwen2.5-7B-Instruct-bnb-4bit |
| A100 | 40 GB | Qwen2.5-14B-Instruct-bnb-4bit |

**Before running**: set `HF_TOKEN` and `HUB_MODEL_ID` in Cell 3.

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
    return r.returncode

# Core HF + quantization
_run('pip install -q --upgrade pip')
_run('pip install -q "transformers>=4.45.0" "accelerate>=0.34.0" "peft>=0.13.0" "datasets>=2.21.0" "huggingface_hub>=0.25.0"')
_run('pip install -q "bitsandbytes>=0.43.0"')

# TRL — GRPO (cap at 0.14 for stable API; notebook handles both 0.12 and 0.15 APIs)
_run('pip install -q "trl>=0.12.0,<0.15.0"')

# Unsloth — efficient LoRA/4-bit
_run('pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"')

# CyberSOC environment dependencies
_run('pip install -q "fastapi>=0.115.0" "uvicorn[standard]>=0.30.0" "pydantic>=2.9.0" "openenv-core" "aiofiles" "networkx" "websockets"')

_run('pip install -q tensorboard')

print('\u2705 All packages installed.')

## Cell 2 — Clone / Locate the CyberSOC Repo

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/Ajayyy00/CyberSOC-upgraded.git'
REPO_DIR = '/content/CyberSOC'

if os.path.isdir(os.path.join(REPO_DIR, 'server')):
    print(f'Repo already present at {REPO_DIR}')
elif os.path.isdir(os.path.join(os.getcwd(), 'server')):
    REPO_DIR = os.getcwd()
    print(f'Using cwd as repo root: {REPO_DIR}')
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    print(f'Cloned to {REPO_DIR}')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print('Repo contents:', [f for f in os.listdir('.') if not f.startswith('.')])

## Cell 3 — Training Configuration
*(edit these before running)*

In [ ]:
import os, torch

# ── User settings ─────────────────────────────────────────────────────────────
HF_TOKEN     = ''   # HuggingFace write token, or leave '' and set env var HF_TOKEN
HUB_MODEL_ID = ''   # e.g. 'yourname/cybersoc-analyst-7b'  (leave '' to skip push)
OUTPUT_DIR   = './checkpoints/cybersoc-grpo'

# ── Model (auto-selected by GPU VRAM) ─────────────────────────────────────────
gpu_mem_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0
print(f'GPU memory: {gpu_mem_gb} GB')

if gpu_mem_gb >= 38:
    MODEL_NAME = 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit'
elif gpu_mem_gb >= 20:
    MODEL_NAME = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
else:
    MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
print(f'Auto-selected model: {MODEL_NAME}')

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.0
MAX_SEQ_LENGTH = 4096
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

# ── GRPO ──────────────────────────────────────────────────────────────────────
MAX_STEPS                   = 200     # raise to 500+ for serious training
NUM_GENERATIONS             = 4       # completions per prompt (reduce to 2 on T4)
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE               = 5e-6
WARMUP_RATIO                = 0.1
MAX_PROMPT_LENGTH           = 1024
MAX_COMPLETION_LENGTH       = 1536
TEMPERATURE                 = 0.9
TOP_P                       = 0.95
BETA                        = 0.001   # KL penalty
LOGGING_STEPS               = 5
SAVE_STEPS                  = 50
SEED                        = 42

# ── Dataset ───────────────────────────────────────────────────────────────────
TASK_IDS         = ['easy', 'medium', 'hard']
PROMPTS_PER_TASK = 10

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

print('Configuration ready.')

## Cell 4 — Verify Environment (In-Process)

No HTTP server needed — `CyberSOCEnvironment` runs directly in Python.
This cell verifies imports and a quick smoke-test reset.

In [ ]:
import sys, os

# Ensure repo root is on sys.path (idempotent)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from server.play_environment import CyberSOCEnvironment
from models import SOCActionWrapper, SOCObservation

def _obs_dict(obs):
    """Convert SOCObservation (Pydantic model) to plain dict."""
    if hasattr(obs, 'model_dump'):
        return obs.model_dump()
    if hasattr(obs, '__dict__'):
        return obs.__dict__
    return dict(obs)

print('Smoke-testing all three task difficulties ...')
for task in ['easy', 'medium', 'hard']:
    _env = CyberSOCEnvironment(fsp_mode=False)
    _obs = _env.reset(task_id=task)
    _d   = _obs_dict(_obs)
    print(f'  [{task:6s}]  alerts={len(_d.get("alert_queue", []))}  '
          f'step={_d.get("step_count", "?")}  ✓')
    del _env

print('\n\u2705 CyberSOCEnvironment is working correctly.')

## Cell 5 — Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

print(f'Loading {MODEL_NAME} ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = True,
    dtype          = None,
    token          = HF_TOKEN or None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                         = LORA_R,
    target_modules            = TARGET_MODULES,
    lora_alpha                = LORA_ALPHA,
    lora_dropout              = LORA_DROPOUT,
    bias                      = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state              = SEED,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params/1e9:.2f}B')
print(f'Trainable params: {trainable_params/1e6:.1f}M  ({100*trainable_params/total_params:.2f}%)')
print('\u2705 Model loaded with LoRA adapters.')

## Cell 6 — Build Training Dataset

In [ ]:
import textwrap
from datasets import Dataset

SYSTEM_PROMPT = textwrap.dedent("""
    You are an expert SOC (Security Operations Center) analyst responding to a live
    cybersecurity incident. Given the current environment state, output a complete JSON
    array of investigation and containment actions, ending with submit_containment_plan.

    AVAILABLE ACTIONS — use exact field names:
      {\"type\": \"correlate_alerts\",          \"alert_ids\": [\"ID1\", \"ID2\"]}
      {\"type\": \"query_host\",                \"hostname\": \"HOSTNAME\"}
      {\"type\": \"run_forensics\",             \"hostname\": \"HOSTNAME\"}
      {\"type\": \"enrich_ioc\",               \"ioc_value\": \"VALUE\", \"ioc_type\": \"ip|domain|hash\"}
      {\"type\": \"scan_host_vulnerabilities\", \"hostname\": \"HOSTNAME\"}
      {\"type\": \"block_ioc\",                \"ioc_value\": \"VALUE\", \"ioc_type\": \"ip|domain|hash\"}
      {\"type\": \"kill_process\",             \"hostname\": \"HOSTNAME\", \"process_name\": \"PROCESS\"}
      {\"type\": \"isolate_segment\",          \"subnet\": \"SUBNET\", \"reason\": \"REASON\"}
      {\"type\": \"trigger_playbook\",         \"playbook_name\": \"NAME\", \"target\": \"HOSTNAME\"}
      {\"type\": \"submit_containment_plan\",  \"plan\": [
          {\"threat_id\": \"T-ID\", \"actions_taken\": [\"action1\"], \"root_cause\": \"...\", \"confidence\": 0.9}
        ], \"executive_summary\": \"TEXT\"}

    PLAYBOOK NAMES: ransomware_containment | c2_disruption | lateral_movement_lockdown |
                    phishing_response | data_exfil_stop

    GRADED ON 10 dimensions (evidence-gated):
      1. TRIAGE      : correlate_alerts on related alert pairs
      2. INVESTIGATE : query_host then run_forensics on every alert source host
      3. ENRICH      : enrich_ioc on IOCs from alerts and forensics
      4. SCAN        : scan_host_vulnerabilities on confirmed-compromised hosts
      5. CONTAIN     : kill malicious processes; block relevant IOCs
      6. ISOLATE     : isolate_segment only for confirmed-compromised subnets
      7. REPORT      : submit_containment_plan as the FINAL action

    Output ONLY a valid JSON array. No markdown fences, no prose.
""").strip()


def _format_observation(obs_dict: dict, task_id: str) -> str:
    alerts    = obs_dict.get('alert_queue', [])
    topo      = obs_dict.get('network_topology', {})
    threats   = obs_dict.get('active_threats', [])
    playbooks = obs_dict.get('available_playbooks', [])

    # Normalize alert dicts (may be Pydantic objects or plain dicts)
    def _alert_dict(a):
        return a.model_dump() if hasattr(a, 'model_dump') else dict(a)

    alert_lines = []
    for a in alerts:
        ad   = _alert_dict(a)
        iocs = ', '.join(ad.get('ioc_indicators', []))
        sev  = ad.get('severity', '?')
        sev  = sev.value.upper() if hasattr(sev, 'value') else str(sev).upper()
        line = (f"  [{sev}] {ad.get('alert_id','?')} | "
                f"{ad.get('source_host','?')} | {ad.get('threat_type','?')} | "
                f"{str(ad.get('description',''))[:90]}")
        if iocs:
            line += f'\n    IOCs: {iocs}'
        alert_lines.append(line)

    topo_d = topo.model_dump() if hasattr(topo, 'model_dump') else dict(topo)
    subnets = ' | '.join(f'{k}:{v}' for k, v in topo_d.get('subnets', {}).items())

    return '\n'.join([
        f'TASK DIFFICULTY : {task_id.upper()}',
        f'MAX STEPS       : {obs_dict.get("max_steps", 30)}',
        '',
        f'ALERT QUEUE ({len(alerts)} alerts):',
        *alert_lines,
        '',
        'NETWORK TOPOLOGY:',
        f'  Hosts: {topo_d.get("total_hosts","?")} total | '
        f'{topo_d.get("compromised_count",0)} compromised | '
        f'{topo_d.get("isolated_count",0)} isolated',
        f'  Subnets: {subnets}',
        '',
        f'ACTIVE THREAT IDs : {", ".join(str(t) for t in threats) if threats else "check alerts"}',
        f'AVAILABLE PLAYBOOKS: {", ".join(playbooks)}',
        '',
        'Generate your complete action sequence as a JSON array:',
    ])


def _build_prompt(obs_dict: dict, task_id: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': _format_observation(obs_dict, task_id)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


rows = []
for task_id in TASK_IDS:
    for i in range(PROMPTS_PER_TASK):
        try:
            _env = CyberSOCEnvironment(fsp_mode=False)
            obs  = _env.reset(task_id=task_id)
            obs_d = _obs_dict(obs)
            rows.append({'prompt': _build_prompt(obs_d, task_id), 'task_id': task_id})
            print(f'  [{task_id}] prompt {i+1}/{PROMPTS_PER_TASK} ✓')
        except Exception as e:
            print(f'  [warn] {task_id} prompt {i+1} failed: {e}')
        finally:
            del _env

if not rows:
    raise RuntimeError('Dataset is empty — check CyberSOCEnvironment imports above.')

dataset = Dataset.from_list(rows)
print(f'\n\u2705 Dataset: {len(dataset)} prompts  |  columns: {dataset.column_names}')
print(f'\nSample prompt (first 600 chars):\n{dataset[0]["prompt"][:600]}...')

## Cell 7 — Define Reward Functions

Each reward function replays the model's completion against a fresh
`CyberSOCEnvironment` instance in-process.  
Scores are read from `reward_dimensions` (live per-step) when available,
with `grade_breakdown` (final, set only when `done=True`) as fallback.

In [ ]:
import json, re
from typing import List

DIMENSION_NAMES = [
    'threat_containment',
    'ioc_blocking',
    'forensic_investigation',
    'siem_correlation',
    'threat_intel_usage',
    'vuln_root_cause',
    'business_impact',
    'step_efficiency',
    'plan_coverage',
    'plan_evidence_quality',
]

WEIGHTS = {
    'threat_containment':     0.20,
    'ioc_blocking':           0.12,
    'forensic_investigation': 0.10,
    'siem_correlation':       0.08,
    'threat_intel_usage':     0.08,
    'vuln_root_cause':        0.08,
    'business_impact':        0.10,
    'step_efficiency':        0.07,
    'plan_coverage':          0.10,
    'plan_evidence_quality':  0.07,
}


def _extract_json_array(text: str):
    """Return the first JSON array found in text, or None."""
    text = text.strip()
    # Strip markdown fences
    m = re.search(r'```(?:json)?\s*([\s\S]+?)```', text)
    if m:
        text = m.group(1).strip()
    # Find first [...] block
    m = re.search(r'(\[[\s\S]+\])', text)
    candidate = m.group(1) if m else text
    try:
        result = json.loads(candidate)
        return result if isinstance(result, list) else None
    except (json.JSONDecodeError, ValueError):
        return None


def _replay_episode(completion: str, task_id: str = 'hard') -> dict:
    """
    Parse completion as a JSON action list and replay it against a fresh env.
    Returns the last observation dict (or an empty dict on failure).
    Uses reward_dimensions for live per-step scores.
    """
    actions = _extract_json_array(completion)
    if not actions:
        return {}

    try:
        env = CyberSOCEnvironment(fsp_mode=False)
        obs = env.reset(task_id=task_id)
        last_obs_dict = _obs_dict(obs)

        for action in actions:
            if not isinstance(action, dict) or 'type' not in action:
                continue
            # Skip red-team action types (should never appear in Blue completions)
            if action['type'] in {'lateral_pivot', 'deploy_payload', 'evade_detection', 'pass_turn'}:
                continue
            try:
                wrapped = SOCActionWrapper(**action)
                obs     = env.step(wrapped)
                last_obs_dict = _obs_dict(obs)
            except Exception:
                continue  # invalid action — skip and keep going
            if last_obs_dict.get('done', False):
                break

        return last_obs_dict
    except Exception:
        return {}
    finally:
        try:
            del env
        except Exception:
            pass


def _read_scores(obs_dict: dict) -> dict:
    """
    Return per-dimension score dict.
    Prefers reward_dimensions (live, always present after first step),
    falls back to grade_breakdown (only set at episode end).
    """
    rd = obs_dict.get('reward_dimensions')
    if rd and isinstance(rd, dict) and any(v != 0 for v in rd.values()):
        return rd
    gb = obs_dict.get('grade_breakdown')
    return gb if gb and isinstance(gb, dict) else {}


def _make_dim_fn(dimension: str):
    def reward_fn(completions: List[str], **kwargs) -> List[float]:
        raw_tids = kwargs.get('task_id', ['hard'] * len(completions))
        if isinstance(raw_tids, str):
            raw_tids = [raw_tids] * len(completions)
        scores = []
        for completion, tid in zip(completions, raw_tids):
            obs = _replay_episode(completion, task_id=tid)
            scores.append(float(_read_scores(obs).get(dimension, 0.0)))
        return scores
    reward_fn.__name__ = f'soc_{dimension}'
    return reward_fn


def _make_weighted_fn():
    def reward_fn(completions: List[str], **kwargs) -> List[float]:
        raw_tids = kwargs.get('task_id', ['hard'] * len(completions))
        if isinstance(raw_tids, str):
            raw_tids = [raw_tids] * len(completions)
        scores = []
        for completion, tid in zip(completions, raw_tids):
            obs = _replay_episode(completion, task_id=tid)
            sd  = _read_scores(obs)
            weighted = sum(sd.get(k, 0.0) * w for k, w in WEIGHTS.items())
            scores.append(float(weighted))
        return scores
    reward_fn.__name__ = 'soc_weighted_total'
    return reward_fn


# 10 per-dimension + 1 weighted composite
reward_fns = [_make_dim_fn(dim) for dim in DIMENSION_NAMES]
reward_fns.append(_make_weighted_fn())

print(f'\u2705 {len(reward_fns)} reward functions registered:')
for fn in reward_fns:
    print(f'   {fn.__name__}')

# ── Sanity check ──────────────────────────────────────────────────────────────
print('\nRunning reward sanity check ...')
_env_sc    = CyberSOCEnvironment(fsp_mode=False)
_obs_sc    = _env_sc.reset(task_id='easy')
_alerts_sc = _obs_sc.alert_queue if hasattr(_obs_sc, 'alert_queue') else _obs_dict(_obs_sc).get('alert_queue', [])

_a0   = _alerts_sc[0]
_id0  = _a0.alert_id if hasattr(_a0, 'alert_id') else _a0['alert_id']
_h0   = _a0.source_host if hasattr(_a0, 'source_host') else _a0['source_host']
_id1  = (_alerts_sc[1].alert_id if len(_alerts_sc) > 1
         else _id0 + '-b')

_test_completion = json.dumps([
    {'type': 'correlate_alerts', 'alert_ids': [_id0, _id1]},
    {'type': 'run_forensics',    'hostname':  _h0},
    {'type': 'submit_containment_plan',
     'plan': [{'threat_id': 'T-EASY-001', 'actions_taken': ['run_forensics'],
               'root_cause': 'test', 'confidence': 0.8}],
     'executive_summary': 'Sanity-check episode.'},
])
del _env_sc

_sc_scores = reward_fns[-1]([_test_completion], task_id=['easy'])
print(f'   soc_weighted_total for test completion = {_sc_scores[0]:.4f}')
assert _sc_scores[0] >= 0, 'Reward function returned negative — check env'
print('\u2705 Reward functions verified.')

## Cell 8 — Configure and Run GRPO Training

In [ ]:
import os
from trl import GRPOTrainer, GRPOConfig

os.makedirs(OUTPUT_DIR, exist_ok=True)

grpo_kwargs = dict(
    output_dir                   = OUTPUT_DIR,
    max_steps                    = MAX_STEPS,
    num_train_epochs             = 1,
    per_device_train_batch_size  = PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps  = GRADIENT_ACCUMULATION_STEPS,
    learning_rate                = LEARNING_RATE,
    warmup_ratio                 = WARMUP_RATIO,
    num_generations              = NUM_GENERATIONS,
    max_prompt_length            = MAX_PROMPT_LENGTH,
    max_completion_length        = MAX_COMPLETION_LENGTH,
    temperature                  = TEMPERATURE,
    top_p                        = TOP_P,
    beta                         = BETA,
    logging_steps                = LOGGING_STEPS,
    save_steps                   = SAVE_STEPS,
    seed                         = SEED,
    bf16                         = True,
    report_to                    = 'tensorboard',
    remove_unused_columns        = False,  # keeps task_id column for reward fns
    dataloader_num_workers       = 0,      # avoids multiprocessing issues in notebooks
)

if HUB_MODEL_ID and HF_TOKEN:
    grpo_kwargs.update(
        push_to_hub  = True,
        hub_model_id = HUB_MODEL_ID,
        hub_strategy = 'checkpoint',
        hub_token    = HF_TOKEN,
    )
    print(f'Hub push enabled \u2192 {HUB_MODEL_ID}')
else:
    grpo_kwargs['report_to'] = 'none'
    print('Hub push disabled (set HUB_MODEL_ID and HF_TOKEN to enable).')

grpo_config = GRPOConfig(**grpo_kwargs)

# TRL ≥0.15 uses processing_class; ≤0.14 uses tokenizer — handle both
try:
    trainer = GRPOTrainer(
        model            = model,
        args             = grpo_config,
        processing_class = tokenizer,
        reward_funcs     = reward_fns,
        train_dataset    = dataset,
    )
    print('Trainer initialised (new API: processing_class).')
except TypeError:
    trainer = GRPOTrainer(
        model         = model,
        tokenizer     = tokenizer,
        reward_funcs  = reward_fns,
        args          = grpo_config,
        train_dataset = dataset,
    )
    print('Trainer initialised (legacy API: tokenizer).')

eff_batch = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * NUM_GENERATIONS
print(f'\n\u2705 Starting GRPO training for {MAX_STEPS} steps ...')
print(f'   Effective batch size : {eff_batch}')
print(f'   Output dir           : {OUTPUT_DIR}\n')

train_result = trainer.train()

print('\n' + '='*60)
print('Training complete!')
print(f'  Steps ran  : {train_result.global_step}')
print(f'  Train loss : {train_result.training_loss:.6f}')
print('='*60)

## Cell 9 — Save Model & Push to Hub

In [ ]:
import os

# Save LoRA adapters
lora_dir = os.path.join(OUTPUT_DIR, 'final-lora')
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f'LoRA adapters saved \u2192 {lora_dir}')

# Optional: save merged 16-bit weights (needed for vLLM inference)
SAVE_MERGED = False
if SAVE_MERGED:
    merged_dir = os.path.join(OUTPUT_DIR, 'final-merged')
    model.save_pretrained_merged(merged_dir, tokenizer, save_method='merged_16bit')
    print(f'Merged 16-bit model saved \u2192 {merged_dir}')

# Push to HF Hub
if HUB_MODEL_ID and HF_TOKEN:
    print(f'\nPushing LoRA adapters to huggingface.co/{HUB_MODEL_ID} ...')
    model.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
    print(f'\u2705 Pushed to https://huggingface.co/{HUB_MODEL_ID}')
else:
    print('Skipping Hub push (HUB_MODEL_ID or HF_TOKEN not set).')

print('\n\u2705 All done.')

## Cell 10 — Cleanup

In [ ]:
import torch, gc

del trainer
gc.collect()
torch.cuda.empty_cache()
print('GPU cache cleared.')